<a href="https://colab.research.google.com/github/helenamrch/OAS5900-Masteroppgave/blob/main/name_context_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Name-in-Context Search

Searches for participant names from `navn_output_test.xlsx` in the `.md` protocol files
of each municipality subfolder. Returns paragraphs where names appear in context
(more than 3 words), filtering out standalone name-only lines.

**Output:** `name_context_matches.csv` with columns: `municipality, name, matched_text, filename`

In [5]:
!git config --global user.email "your.email@example.com"
!git config --global user.name "Your Name"

!git init

# Remove existing remote if it exists to avoid 'remote origin already exists' error
!git remote remove origin || true

# Add remote with Personal Access Token (PAT) for authentication
# Replace YOUR_GITHUB_PAT with your actual Personal Access Token
!git remote add origin https://YOUR_GITHUB_PAT@github.com/helenamrch/OAS5900-Masteroppgave.git

# Ensure a commit exists before pushing.
# Create or update README.md to ensure there's content to commit.
!echo "Initial project setup for Colab." > README.md
!git add README.md
!git commit -m "Initial commit from Colab" || true # '|| true' allows it to pass if no changes

# Rename the current branch to 'main' (if not already 'main')
!git branch -M main

# Push to the remote repository
!git push -u origin main

Reinitialized existing Git repository in /content/.git/
[main (root-commit) eab6bd3] Initial commit from Colab
 1 file changed, 1 insertion(+)
 create mode 100644 README.md
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
# ── Cell 1 ─ Imports & configuration ────────────────────────────────────────
import re
import unicodedata
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"/content/drive/MyDrive/Bufdir_ungmedvirk/samlet_ungrad_protokoller")
INPUT_XLSX = BASE_DIR / "Alle_navn_fra_alle_protokoller_ med_kjonn.xlsx"
OUTPUT_CSV = BASE_DIR / "name_context_matches_ny_yuri.csv"

print(f"Base dir : {BASE_DIR}")
print(f"Input    : {INPUT_XLSX}")
print(f"Output   : {OUTPUT_CSV}")

Base dir : /content/drive/MyDrive/Bufdir_ungmedvirk/samlet_ungrad_protokoller
Input    : /content/drive/MyDrive/Bufdir_ungmedvirk/samlet_ungrad_protokoller/Alle_navn_fra_alle_protokoller_ med_kjonn.xlsx
Output   : /content/drive/MyDrive/Bufdir_ungmedvirk/samlet_ungrad_protokoller/name_context_matches_ny_yuri.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Cell 2 ─ Load input & deduplicate ────────────────────────────────────────

raw = pd.read_excel(INPUT_XLSX, engine='openpyxl')
print(f"Raw rows: {len(raw)}")

# Keep only municipality + name, deduplicate
pairs = raw[['municipality', 'name']].dropna().drop_duplicates()
pairs = pairs.reset_index(drop=True)

# Normalize Unicode (NFC) so folder names match
pairs['municipality'] = pairs['municipality'].apply(
    lambda s: unicodedata.normalize('NFC', s)
)
pairs['name'] = pairs['name'].apply(
    lambda s: unicodedata.normalize('NFC', s)
)

print(f"Unique (municipality, name) pairs: {len(pairs)}")
print(f"Municipalities: {pairs['municipality'].nunique()}")
pairs.head(10)

Raw rows: 7391
Unique (municipality, name) pairs: 1906
Municipalities: 58


,municipality,name
0,Aukra,Abdalle Mahamed Abdi
1,Nesodden,Abdinasir Ibrahim
2,Vestfold fylkeskommune,Abdi-Rahin Ise Mohamed
3,Vestfold fylkeskommune,Abdi-Rahin Lognvik-Vartdal
4,Aukra,Abdirahman Mohamed Abdi
5,Kvinnherad,Abdisalem Zekeriye
6,Lørenskog,Abdullha Khaliq
7,Gildeskål,Abel Grovassbakk
8,Gjerderum,Ada Chioma Owa
9,Løten,Ada Elise


In [ ]:
# ── Cell 3 ─ Search names in .md files ────────────────────────────────────────

def read_file(path):
    """Read a file, trying UTF-8 first then Latin-1."""
    try:
        return path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        return path.read_text(encoding='latin-1')


def extract_paragraphs(text):
    """
    Split text into paragraphs (separated by one or more blank lines).
    Strip markdown heading markers but keep heading text.
    """
    raw_paras = re.split(r'\n\s*\n', text)
    result = []
    for p in raw_paras:
        # Strip markdown headings
        cleaned = re.sub(r'^#+\s*', '', p.strip(), flags=re.MULTILINE)
        # Collapse internal whitespace
        cleaned = ' '.join(cleaned.split())
        if cleaned:
            result.append(cleaned)
    return result


# Group names by municipality for efficient processing
grouped = pairs.groupby('municipality')['name'].apply(list).to_dict()

results = []
municipalities_processed = 0
municipalities_missing = []

for muni, names in sorted(grouped.items()):
    muni_dir = BASE_DIR / muni
    if not muni_dir.is_dir():
        municipalities_missing.append(muni)
        continue

    md_files = sorted(muni_dir.glob('*.md'))
    if not md_files:
        continue

    # Pre-compile regex patterns for each name: full name + first name
    name_patterns = []
    for name in names:
        full_pat = re.compile(r'\b' + re.escape(name) + r'\b', re.IGNORECASE)
        first_name = name.split()[0]
        first_pat = re.compile(r'\b' + re.escape(first_name) + r'\b', re.IGNORECASE)
        name_patterns.append((name, full_pat, first_pat))

    for md_file in md_files:
        text = read_file(md_file)
        # Normalize Unicode in file content
        text = unicodedata.normalize('NFC', text)
        paragraphs = extract_paragraphs(text)

        for para in paragraphs:
            # Skip short paragraphs (3 words or fewer)
            if len(para.split()) <= 3:
                continue

            for name, full_pat, first_pat in name_patterns:
                if full_pat.search(para) or first_pat.search(para):
                    results.append({
                        'municipality': muni,
                        'name': name,
                        'matched_text': para,
                        'filename': md_file.name,
                    })

    municipalities_processed += 1
    if municipalities_processed % 10 == 0:
        print(f"  Processed {municipalities_processed} municipalities, {len(results)} matches so far...")

print(f"\nDone: {municipalities_processed} municipalities, {len(results)} total matches")
if municipalities_missing:
    print(f"WARNING – folders not found: {municipalities_missing}")

  Processed 10 municipalities, 2319 matches so far...
  Processed 20 municipalities, 5047 matches so far...
  Processed 30 municipalities, 7573 matches so far...
  Processed 40 municipalities, 9679 matches so far...
  Processed 50 municipalities, 17086 matches so far...

Done: 58 municipalities, 20452 total matches


In [ ]:
# ── Cell 4 ─ Save CSV ─────────────────────────────────────────────────────────

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"Saved: {OUTPUT_CSV}")
print(f"Total matches    : {len(df)}")
print(f"Unique names     : {df['name'].nunique()}")
print(f"Municipalities   : {df['municipality'].nunique()}")
print(f"Files with hits  : {df['filename'].nunique()}")
print()
print("Matches per municipality:")
print(df.groupby('municipality').size().sort_values(ascending=False).to_string())
print()
df.head(20)

NameError: name 'results' is not defined